In [26]:
!pip install -q qiskit
!pip install -q qiskit-algorithms

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 15.4 MB/s eta 0:00:00


In [28]:
from qiskit import QuantumCircuit
from qiskit_algorithms.optimizers import L_BFGS_B

In [17]:
#Preprocess Dataset
def preprocess_data(X, encoding_mode='angle', pad=False):
    X_normalized = X
    n_features = X.shape[1]
    n_qubits = n_features
    if encoding_mode == 'amplitude':
        n_qubits = np.ceil(np.log2(n_features)).astype(int)
        next_power_of_two = 2 ** n_qubits
        if pad:
            #Pad to power of two
            #Pad each ft. vector with 0s to match the nearest sqr
            resized_features = \
                np.pad(X, ((0, 0),
                           (0, next_power_of_two - n_features)), 'constant')
        else:
            #Interpolate the data
            def resize_to_power_of_two(features, new_size):
                original_size = len(features)
                x_old = np.linspace(0, 1, original_size)
                x_new = np.linspace(0, 1, new_size)
                resized_features = np.interp(x_new, x_old, features)
                return resized_features
            resized_features = \
                np.array([
                    resize_to_power_of_two(
                        sample, next_power_of_two) for sample in X])
        #Normal. each samp. to have an norm 1 in L2 (for amp encoding)
        X_normalized = normalize(resized_features, norm='l2')
    elif encoding_mode == 'angle':
        #Standardize the data (mean = 0, std = 1)
        #For angle encoding
        scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
        X_normalized = scaler.fit_transform(X)
    else:
        raise ValueError('Unsuported encoding mode.')
    return X_normalized, n_qubits

In [29]:
#VQC creation
def create_vqc_circuit(num_qubits, params, feature_map:QuantumCircuit,
                       ansatz, features, obs):
    qc = QuantumCircuit(num_qubits)
    circ_params = {}
    if feature_map is None:
        qc.initialize(features)
    else:
      # Assign feature map parameters
        for p, value in zip(feature_map.parameters, features):
            circ_params[p] = value
        qc = qc.compose(feature_map)
    for p, value in zip(ansatz.parameters, params):
        circ_params[p] = value
    #var_cirq.assign_parameters(circ_params, inplace=True)
    qc = qc.compose(ansatz)
    # Transpile classifier
    transpiled_classifier = pm.run(qc)
    # Transpile observable
    transpiled_obs = obs.apply_layout(layout=transpiled_classifier.layout)
    return (transpiled_classifier, transpiled_obs, circ_params)

In [31]:
#Training the VQC
def train_vqc(X_train, y_train, num_qubits, backend, feature_map,
              ansatz, optimizer, obs, callback=None):

    # Parâmetros iniciais aleatórios
    initial_params = np.random.uniform(0, 2 * np.pi, ansatz.num_parameters)

    # Define a função de custo
    def objective_function(params):
        cost = cost_function(
            params,
            X_train,
            y_train,
            num_qubits,
            backend,
            feature_map=feature_map,
            ansatz=ansatz,
            obs=obs
        )
        if callback is not None:
            callback(params, cost)
        return cost

    # Executa o otimizador
    print("Starting optimization...")
    result = optimizer.minimize(fun=objective_function, x0=initial_params)
    print(f"Optimization completed. | Final cost: {result.fun}")

    return result

optimizer = L_BFGS_B(maxiter=100)

In [32]:
#Testing the VQC
def test_vqc(X_test, optimal_params, num_qubits,
feature_map, ansatz, backend, obs, num_shots=1024):
    predictions = []
    for features in X_test:
        pub = create_vqc_circuit(
        num_qubits, optimal_params, feature_map, ansatz, features, obs=obs)
        job = estimator.run([pub])
        result = job.result()[0].data.evs
        predictions.append(result)
    return np.array(predictions)

In [33]:
#Building the classifier with quantum methods
def run_experiment(X, y, k_fold, ansatze, ansatz_reps_list, encoding_mode_list,
obs_list, optimizer_type_list, num_iteration_list, tol=1e-8):
    kf = KFold(n_splits=k_fold, shuffle=True, random_state=42)
    performance_list = []
    metrics_summary = []
    for ansatz_type in ansatze:
        for ansatz_reps in ansatz_reps_list:
            for encoding_mode in encoding_mode_list:
                for obs in obs_list:
                    for optimizer_type in optimizer_type_list:
                        for num_iteration in num_iteration_list:
                            performance_sum = 0
                            current_k = 0
                            fold_metrics = {'accuracy': [],
                                            'precision': [],
                                            'recall': [],
                                            'f1': []
                            }
                            for train_index, test_index in kf.split(X):
                                current_k += 1
                                X_train, X_test = X[train_index], X[test_index]
                                y_train, y_test = y[train_index], y[test_index]
                                print(f'K={current_k}|Train set size:{X_train.shape}, test set size:{X_test.shape}')
                                X_train_scaled, n_qubits = preprocess_data(X_train, encoding_mode)
                                X_test_scaled, n_qubits = preprocess_data(X_test, encoding_mode)
                                ansatz = ansatz_type(n_qubits, reps=ansatz_reps)
                                if optimizer_type == SPSA:
                                    optimizer = optimizer_type(maxiter=num_iteration)
                                else:
                                    optimizer = optimizer_type(maxiter=num_iteration, tol=tol)
                                num_var_params = X_test_scaled.shape[1]
                                feature_map = build_feature_map(num_qubits=n_qubits,
                                num_var_params=num_var_params, encoding_mode=encoding_mode)
                                result = train_vqc(X_train_scaled,
                                y_train,
                                num_qubits=n_qubits,
                                backend=backend,
                                feature_map=feature_map,
                                ansatz=ansatz,
                                obs=obs,
                                optimizer=optimizer,
                                callback=callback_graph
                                )
                                predictions = test_vqc(X_test_scaled,
                                optimal_params=result.x,
                                num_qubits=n_qubits,
                                backend=backend,
                                feature_map=feature_map,
                                ansatz=ansatz,
                                obs=obs
                                )
                                # Convert predictions to binary labels (0 ou 1)
                                pred_labels = (predictions > 0.5).astype(int)
                                # Calcular métricas
                                acc = accuracy_score(y_test, pred_labels)
                                prec = precision_score(y_test, pred_labels, zero_division=0)
                                rec = recall_score(y_test, pred_labels)
                                f1 = f1_score(y_test, pred_labels)
                                # Armazenar métricas para esta fold
                                fold_metrics['accuracy'].append(acc)
                                fold_metrics['precision'].append(prec)
                                fold_metrics['recall'].append(rec)
                                fold_metrics['f1'].append(f1)
                                performance_sum += acc # Acumular acurácia para desempenho geral
                            performance_avg = performance_sum / k_fold
                            performance_list.append(performance_avg)
                            metrics_summary.append(fold_metrics)
    return performance_list, metrics_summary

In [34]:
#Running experiment 1
experiment_result = run_experiment(
X.to_numpy(),
y.to_numpy(),
k_fold = 10,
ansatze=[RealAmplitudes],
ansatz_reps_list=[1],
encoding_mode_list = ['amplitude','phase'],
obs_list = [SparsePauliOp("ZZZIIII")],
optimizer_type_list=[COBYLA],
num_iteration_list=[100]
)

NameError: name 'X' is not defined

In [ ]:
#Running experiment 2
experiment_result = run_experiment(
X.to_numpy(),
y.to_numpy(),
k_fold = 10,
ansatze=[RealAmplitudes],
ansatz_reps_list=[1],
encoding_mode_list = ['amplitude','phase'],
obs_list = [SparsePauliOp("ZZZIIII")],
optimizer_type_list=[COBYLA],
num_iteration_list=[100]
)

In [ ]:
#Running experiment 3
experiment_result = run_experiment(
X.to_numpy(),
y.to_numpy(),
k_fold = 10,
ansatze=[RealAmplitudes, EfficientSU2, TwoLocal],
ansatz_reps_list=[1, 3, 5],
encoding_mode_list = ['amplitude','phase'],
obs_list = [SparsePauliOp("ZZZZZZZ")],
optimizer_type_list=[COBYLA, SPSA, ADAM],
num_iteration_list=[25, 50, 100]
)